# Gemini OCR 핸즈온: GTIN/UPC 추출 및 상품 정보 기록

Google Gemini 모델을 활용하여 상품 이미지에서 바코드(GTIN, UPC 등)를 인식하고,
상품명·상세 정보를 자동으로 추출하는 핸즈온 노트북입니다.

## 목차
1. Setup
2. 공통 함수 정의
3. Image Captioning
4. OCR — GTIN/UPC 추출 및 상품 정보 기록

---
## 1. Setup

필요한 라이브러리를 설치하고 Gemini 클라이언트를 초기화합니다.

In [ ]:
!pip install -q google-genai ultralytics Pillow

In [ ]:
import json

import cv2
from google import genai
from google.genai import types
from PIL import Image
from ultralytics.utils.downloads import safe_download
from ultralytics.utils.plotting import Annotator, colors

In [ ]:
# Vertex AI API 키 기반 Gemini 클라이언트 초기화
# API 키 발급: Google Cloud Console > API 및 서비스 > 사용자 인증 정보 > API 키 만들기
# Vertex AI API가 프로젝트에서 활성화되어 있어야 합니다.
API_KEY = ""  # TODO: 본인의 Vertex AI API 키 입력

client = genai.Client(
    vertexai=True,
    api_key=API_KEY,
)

MODEL_ID = "gemini-3-flash-preview"

---
## 2. 공통 함수 정의

In [ ]:
def inference(image, prompt, temp=0.5):
    """Gemini 모델로 이미지 + 텍스트 프롬프트 추론을 수행합니다."""
    response = client.models.generate_content(
        model=MODEL_ID,
        contents=[prompt, image],
        config=types.GenerateContentConfig(temperature=temp),
    )
    return response.text


def read_image(filename):
    """이미지를 읽어 PIL Image와 너비/높이를 반환합니다."""
    image = cv2.cvtColor(cv2.imread(filename), cv2.COLOR_BGR2RGB)
    h, w = image.shape[:2]
    return Image.fromarray(image), w, h


def clean_results(results):
    """Markdown 코드블록 포맷을 제거하여 JSON 파싱 가능하게 정리합니다."""
    return results.strip().removeprefix("```json").removesuffix("```").strip()

---
## 3. Image Captioning

Gemini 모델을 사용하여 이미지의 내용을 자연어로 설명하는 캡션을 생성합니다.
상품 이미지에 대해 어떤 상품인지, 외관 특징은 무엇인지 파악하는 데 활용할 수 있습니다.

In [ ]:
# 테스트 이미지 다운로드 (원하는 상품 이미지로 교체 가능)
safe_download(
    "https://github.com/ultralytics/notebooks/releases/download/v0.0.0/gemini-image4.jpg"
)

caption_image, _, _ = read_image("gemini-image4.jpg")
display(caption_image)

In [ ]:
caption_prompt = """
What's inside the image, generate a detailed captioning in the form of short
story, Make 4-5 lines and start each sentence on a new line.
"""

print(inference(caption_image, caption_prompt))

### 3-1. 상품 이미지 캡셔닝 (한국어)

상품 이미지에 대해 한국어로 캡션을 생성해 봅니다.

In [ ]:
product_caption_prompt = """
이 이미지에 있는 상품을 한국어로 상세하게 설명해 주세요.
다음 항목을 포함해 주세요:
- 상품 종류
- 외관 특징 (색상, 형태, 크기 등)
- 포장에 보이는 텍스트나 브랜드
- 기타 눈에 띄는 특징
"""

print(inference(caption_image, product_caption_prompt))

---
## 4. OCR — GTIN/UPC 추출 및 상품 정보 기록

Gemini 모델의 OCR 기능을 활용하여 상품 이미지에서 바코드(GTIN, UPC, EAN 등)를 감지하고,
해당 코드를 기반으로 상품명, 설명 등 상세 정보를 추출합니다.

In [ ]:
# 바코드 테스트 이미지 다운로드 (원하는 상품 바코드 이미지로 교체 가능)
safe_download(
    "https://free-barcode.com/barcode/barcode-technology/detail-gtin-14-barcode/5.jpg",
    file="barcode_image.jpg",
)

ocr_image, w, h = read_image("barcode_image.jpg")
display(ocr_image)

In [ ]:
ocr_prompt = """
Extract the barcode and GTIN from the image.
For each detected text area, provide the 'box_2d' and the 'label'.
And based on GTIN, please write down details for the product including name, description, etc.
"""

output_format = """
Return the results strictly in JSON format as a list of objects,
each containing 'box_2d' (y1, x1, y2, x2) and 'label'. No additional text.
"""

results = inference(ocr_image, ocr_prompt + output_format)

try:
    cln_results = json.loads(clean_results(results))
    annotator = Annotator(ocr_image)

    for idx, item in enumerate(cln_results):
        if "box_2d" not in item or "label" not in item:
            continue

        y1, x1, y2, x2 = item["box_2d"]
        y1 = y1 / 1000 * h
        x1 = x1 / 1000 * w
        y2 = y2 / 1000 * h
        x2 = x2 / 1000 * w

        if x1 > x2:
            x1, x2 = x2, x1
        if y1 > y2:
            y1, y2 = y2, y1

        annotator.box_label(
            [x1, y1, x2, y2], label=item["label"], color=colors(idx, True)
        )

    display(Image.fromarray(annotator.result()))

except json.JSONDecodeError as e:
    print(f"JSON parsing failed: {e}")
    print("Raw output from model:")
    print(results)

### 4-1. GTIN/UPC 기반 상품 정보 상세 추출

바코드 영역 감지와 별도로, 상품 정보를 구조화된 JSON으로 추출합니다.

In [ ]:
product_info_prompt = """
이 이미지에서 바코드, GTIN, UPC, EAN 등 모든 상품 식별 코드를 찾아 주세요.
그리고 각 코드에 대해 다음 정보를 JSON 형식으로 반환해 주세요:

{
  "barcode_type": "GTIN-14 / UPC-A / EAN-13 등",
  "barcode_value": "숫자 코드",
  "product_name": "상품명 (알 수 있는 경우)",
  "product_description": "상품 설명",
  "brand": "브랜드명 (알 수 있는 경우)",
  "category": "상품 카테고리",
  "additional_text": "이미지에서 읽은 기타 텍스트"
}

여러 개의 바코드가 있으면 리스트로 반환해 주세요.
JSON만 반환하고 다른 텍스트는 포함하지 마세요.
"""

product_results = inference(ocr_image, product_info_prompt)

try:
    product_info = json.loads(clean_results(product_results))
    print(json.dumps(product_info, indent=2, ensure_ascii=False))
except json.JSONDecodeError as e:
    print(f"JSON parsing failed: {e}")
    print("Raw output from model:")
    print(product_results)

### 4-2. 나만의 상품 이미지로 테스트

아래 셀에서 `image_path`를 변경하여 본인의 상품 이미지를 테스트해 보세요.

In [ ]:
# image_path = "your_product_image.jpg"  # TODO: 본인의 이미지 경로로 변경
# my_image, w, h = read_image(image_path)
# display(my_image)
#
# # 캡셔닝
# print("=== Image Caption ===")
# print(inference(my_image, product_caption_prompt))
#
# # GTIN/UPC 추출
# print("\n=== Product Info ===")
# product_results = inference(my_image, product_info_prompt)
# try:
#     product_info = json.loads(clean_results(product_results))
#     print(json.dumps(product_info, indent=2, ensure_ascii=False))
# except json.JSONDecodeError:
#     print(product_results)